In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/Civil Comments Project'
DATA_DIR    = os.path.join(DRIVE_BASE, 'data')
FIGURES_DIR = os.path.join(DRIVE_BASE, 'outputs/figures')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import arviz as az

print("Setup complete.")

In [ ]:
# Load scored dataset
df = pd.read_csv(os.path.join(DATA_DIR, 'civil_comments_with_mf_scores.csv'))

# Load posterior traces
trace_hurdle    = az.from_netcdf(os.path.join(DATA_DIR, '03_trace_hurdle.nc'))
trace_intensity = az.from_netcdf(os.path.join(DATA_DIR, '03_trace_intensity.nc'))

foundation_names = ['care', 'fairness', 'loyalty', 'authority', 'purity']
vice_cols = [f'mf_{f}_vice' for f in foundation_names]

print(f"Dataset: {len(df):,} rows")
print(f"Traces loaded successfully.")

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#FAFAFA')

gs = gridspec.GridSpec(
    3, 3,
    figure=fig,
    hspace=0.45,
    wspace=0.35,
    left=0.07, right=0.97,
    top=0.91, bottom=0.07
)

colors = {
    'care':      '#E07B54',
    'fairness':  '#5B8DB8',
    'loyalty':   '#6AAB6E',
    'authority': '#9B6BB5',
    'purity':    '#C4A644'
}

# Panel A: Toxicity distribution
ax_a = fig.add_subplot(gs[0, 0])
ax_a.hist(df['toxicity'], bins=40, color='#888888', edgecolor='white', linewidth=0.3)
ax_a.set_title('A. Toxicity Score Distribution\n(74.8% are exactly zero)', fontsize=10, fontweight='bold')
ax_a.set_xlabel('Toxicity Score')
ax_a.set_ylabel('Count')
ax_a.axvline(0, color='crimson', linestyle='--', linewidth=1.2,
             label='75% exactly zero')
ax_a.spines[['top', 'right']].set_visible(False)

# Panel B: Correlation heatmap
ax_b = fig.add_subplot(gs[0, 1:])
corr_data = df[vice_cols].corrwith(df['toxicity'])
corr_data.index = [n.capitalize() for n in foundation_names]
bars = ax_b.barh(
    corr_data.index,
    corr_data.values,
    color=[colors[n] for n in foundation_names],
    edgecolor='white',
    height=0.6
)
ax_b.axvline(0, color='black', linewidth=0.8)
ax_b.set_title('B. Pearson Correlation: Moral Vice Scores → Toxicity',
               fontsize=10, fontweight='bold')
ax_b.set_xlabel('Correlation (r)')
ax_b.spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, corr_data.values):
    ax_b.text(val + 0.003, bar.get_y() + bar.get_height()/2,
              f'{val:.3f}', va='center', fontsize=9)

# Panel C: Forest plot Part 1
ax_c = fig.add_subplot(gs[1, :2])
means_h = trace_hurdle.posterior['beta_hurdle'].values.mean(axis=(0, 1))
lo_h    = np.percentile(trace_hurdle.posterior['beta_hurdle'].values, 3,  axis=(0, 1))
hi_h    = np.percentile(trace_hurdle.posterior['beta_hurdle'].values, 97, axis=(0, 1))

y_pos = range(len(foundation_names))
ax_c.barh(y_pos, means_h,
          xerr=[means_h - lo_h, hi_h - means_h],
          color=[colors[n] for n in foundation_names],
          alpha=0.85, capsize=5, height=0.5, edgecolor='white')
ax_c.axvline(0, color='black', linewidth=1, linestyle='--')
ax_c.set_yticks(y_pos)
ax_c.set_yticklabels([n.capitalize() for n in foundation_names], fontsize=10)
ax_c.set_xlabel('Posterior Mean Effect (standardized)')
ax_c.set_title('C. Bayesian Hurdle Model — Part 1\nDoes moral-vice language predict crossing the toxicity threshold?',
               fontsize=10, fontweight='bold')
ax_c.spines[['top', 'right']].set_visible(False)

# Panel D: Forest plot Part 2
ax_d = fig.add_subplot(gs[1, 2])
means_i = trace_intensity.posterior['beta_intensity'].values.mean(axis=(0, 1))
lo_i    = np.percentile(trace_intensity.posterior['beta_intensity'].values, 3,  axis=(0, 1))
hi_i    = np.percentile(trace_intensity.posterior['beta_intensity'].values, 97, axis=(0, 1))

ax_d.barh(y_pos, means_i,
          xerr=[means_i - lo_i, hi_i - means_i],
          color=[colors[n] for n in foundation_names],
          alpha=0.85, capsize=5, height=0.5, edgecolor='white')
ax_d.axvline(0, color='black', linewidth=1, linestyle='--')
ax_d.set_yticks(y_pos)
ax_d.set_yticklabels([n.capitalize() for n in foundation_names], fontsize=10)
ax_d.set_xlabel('Posterior Mean Effect (standardized)')
ax_d.set_title('D. Bayesian Hurdle Model — Part 2\nGiven toxic, how intense?',
               fontsize=10, fontweight='bold')
ax_d.spines[['top', 'right']].set_visible(False)

# Panel E: Key findings text box
ax_e = fig.add_subplot(gs[2, :])
ax_e.axis('off')

findings = (
    "KEY FINDINGS\n\n"
    "1.  Purity & Authority are the strongest predictors of a comment crossing the toxicity threshold (Part 1). "
    "Disgust and defiance framing reliably signal toxic content.\n\n"
    "2.  Authority dominates toxicity intensity (Part 2). Once a comment is toxic, defiance-of-authority language "
    "predicts how extreme it becomes — more than any other foundation.\n\n"
    "3.  Loyalty shows a consistent negative effect in both model components, the only foundation to do so. "
    "Comments using loyalty/betrayal framing are less likely to be toxic and less intense when flagged.\n\n"
    "4.  Care and Fairness predict threshold-crossing but not intensity, suggesting cruelty and unfairness framing "
    "marks a comment as harmful without determining its severity.\n\n"
    "5.  A standard regression would have obscured findings 3 and 4 by averaging across both processes. "
    "The two-part hurdle model was necessary to reveal the distinct roles of each moral foundation."
)

ax_e.text(
    0.01, 0.95, findings,
    transform=ax_e.transAxes,
    fontsize=9.5,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.8', facecolor='#EEF2F7', edgecolor='#AABBCC', linewidth=1.2),
    linespacing=1.6
)

# Title
fig.suptitle(
    'Do Moral Foundations Predict Online Toxicity?\n'
    'A Bayesian Hurdle Model Analysis of 10,000 Civil Comments',
    fontsize=14, fontweight='bold', y=0.97
)

save_path = os.path.join(FIGURES_DIR, '04_master_summary.png')
plt.savefig(save_path, dpi=180, bbox_inches='tight', facecolor='#FAFAFA')
print(f"Saved: {save_path}")
plt.show()